In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal as spsig
from pathlib import Path

In [ ]:
plt.rcParams.update({
    "figure.dpi": 300,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.facecolor": "white",
    "font.family": "sans-serif",
})

DATA_ROOT = Path("data/rsvp/sub-01")
SFREQ = 250
PRE_S = 0.200
POST_S = 0.800
PRE_SAMP = int(PRE_S * SFREQ)
CHANNELS = ["O1", "O2", "T5", "P3", "Pz", "P4", "T6", "REF"]
TIME = np.linspace(-PRE_S, POST_S, int((PRE_S + POST_S) * SFREQ))

In [ ]:
sessions = {}
for d in sorted(DATA_ROOT.glob("ses-*")):
    meta = pd.read_csv(d / "metadata.csv")
    meta["session"] = d.name
    sessions[d.name] = {"epochs": np.load(d / "eeg_epochs.npy"), "meta": meta}

all_meta = pd.concat([s["meta"] for s in sessions.values()], ignore_index=True)
all_epochs = np.concatenate([s["epochs"] for s in sessions.values()], axis=0)
all_epochs

In [ ]:
def assign_labels(meta_df):
    t = meta_df[meta_df["is_target"] == 1].copy()
    rts = t.loc[t["response"] == "hit", "rt"].dropna()
    thresh = rts.mean() + 1.5 * (rts.std() if len(rts) > 1 else 0)
    def _label(row):
        if row["response"] == "miss": return 1
        if row["response"] == "hit": return 1 if row["rt"] > thresh else 0
        return np.nan
    t["zoned_out"] = t.apply(_label, axis=1)
    return t

In [ ]:
labeled = pd.concat([assign_labels(s["meta"].assign(session=sid))
                      for sid, s in sessions.items()], ignore_index=True)
labeled = labeled.dropna(subset=["zoned_out"])
labeled["zoned_out"] = labeled["zoned_out"].astype(int)

In [ ]:
foc_list, zon_list = [], []
for sid, s in sessions.items():
    mask = labeled["session"] == sid
    for _, row in labeled[mask].iterrows():
        idx = int(row["index"])
        if idx < len(s["epochs"]):
            (foc_list if row["zoned_out"] == 0 else zon_list).append(s["epochs"][idx])
foc_epochs = np.array(foc_list)
zon_epochs = np.array(zon_list)
zon_epochs, foc_epochs

In [29]:
def bp_filter(data, lo=1.0, hi=20.0, fs=SFREQ):
    b, a = spsig.butter(4, [lo / (fs/2), hi / (fs/2)], btype="band")
    if data.ndim == 1: return spsig.filtfilt(b, a, data)
    return np.array([spsig.filtfilt(b, a, ch) for ch in data])

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
hit_rts = all_meta.loc[(all_meta["response"] == "hit") & all_meta["rt"].notna(), "rt"] * 1000
mu, sigma = hit_rts.mean(), hit_rts.std()
thresh = mu + 1.5 * sigma
ax1.hist(hit_rts, bins=25, color="#4C9F70", edgecolor="white", alpha=0.85)
ax1.axvline(mu, color="k", ls="--", lw=1.5, label=f"Mean = {mu:.0f} ms")
ax1.axvline(thresh, color="#C0392B", ls="--", lw=1.8,
            label=f"Slow threshold = {thresh:.0f} ms\n(mean + 1.5 SD)")
ax1.set_xlabel("Reaction Time (ms)")
ax1.set_ylabel("Count")
ax1.set_title("RT Distribution (Hit Trials)")
ax1.legend(fontsize=9)
targ = all_meta[all_meta["is_target"] == 1]
ses_ids = sorted(targ["session"].unique())
hr = [targ[targ["session"] == s]["response"].eq("hit").mean() for s in ses_ids]
mr = [targ[targ["session"] == s]["response"].eq("miss").mean() for s in ses_ids]
x = np.arange(len(ses_ids))
ax2.bar(x - 0.17, hr, 0.34, label="Hit Rate", color="#4C9F70")
ax2.bar(x + 0.17, mr, 0.34, label="Miss Rate", color="#C0392B")
ax2.set_xticks(x)
ax2.set_xticklabels([s.replace("ses-0", "S") for s in ses_ids])
ax2.set_ylabel("Rate")
ax2.set_xlabel("Session")
ax2.set_title("Target Detection Rate per Session")
ax2.set_ylim(0, 1.05)
ax2.legend()

plt.tight_layout()
plt.savefig("figures/fig1_behavioral.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
n_foc = (labeled["zoned_out"] == 0).sum()
n_miss = (labeled["response"] == "miss").sum()
n_slow = ((labeled["zoned_out"] == 1) & (labeled["response"] == "hit")).sum()
n_zon = n_miss + n_slow
bars = ax1.bar(["Focused", "Zoned-out"], [n_foc, n_zon], color=["#4C9F70", "#C0392B"], width=0.5)
for bar, val in zip(bars, [n_foc, n_zon]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             str(val), ha="center", fontweight="bold", fontsize=12)
ax1.set_ylabel("Number of Trials")
ax1.set_title(f"Class Balance (n = {n_foc + n_zon})")
ax1.set_ylim(0, max(n_foc, n_zon) * 1.15)
bars2 = ax2.bar(["Focused\n(normal-speed hit)", "Zoned-out\n(missed target)", "Zoned-out\n(slow hit)"],
                [n_foc, n_miss, n_slow], color=["#4C9F70", "#C0392B", "#E67E73"], width=0.5)
for bar, val in zip(bars2, [n_foc, n_miss, n_slow]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(val), ha="center", fontweight="bold", fontsize=11)
ax2.set_ylabel("Number of Trials")
ax2.set_title("Label Source Breakdown")
ax2.set_ylim(0, max(n_foc, n_miss, n_slow) * 1.2)
plt.tight_layout()
plt.savefig("figures/fig2_class_distribution.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Imbalance ratio: {n_foc / n_zon:.1f} : 1 (focused : zoned-out)")

In [ ]:
foc_filt = np.array([bp_filter(ep) for ep in foc_epochs])
zon_filt = np.array([bp_filter(ep) for ep in zon_epochs])
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=True, sharey=True)
for ch_idx, (ax, ch_name) in enumerate(zip(axes.flatten(), CHANNELS)):
    foc_mean = foc_filt[:, ch_idx, :].mean(axis=0)
    foc_sem = foc_filt[:, ch_idx, :].std(axis=0) / np.sqrt(len(foc_filt))
    zon_mean = zon_filt[:, ch_idx, :].mean(axis=0)
    zon_sem = zon_filt[:, ch_idx, :].std(axis=0) / np.sqrt(len(zon_filt))
    ax.fill_between(TIME, foc_mean - foc_sem, foc_mean + foc_sem, color="#4C9F70", alpha=0.25)
    ax.fill_between(TIME, zon_mean - zon_sem, zon_mean + zon_sem, color="#C0392B", alpha=0.25)
    ax.plot(TIME, foc_mean, color="#4C9F70", lw=1.8, label=f"Focused (n={len(foc_filt)})")
    ax.plot(TIME, zon_mean, color="#C0392B", lw=1.8, label=f"Zoned-out (n={len(zon_filt)})")
    ax.axvspan(0.3, 0.5, alpha=0.10, color="#7B68EE")
    ax.axvline(0, color="gray", ls="--", lw=0.8)
    ax.axhline(0, color="gray", lw=0.4)
    ax.set_title(ch_name, fontweight="bold")
    ax.set_xlim(-0.2, 0.8)
    ax.grid(True, alpha=0.15)
    if ch_idx >= 4:
        ax.set_xlabel("Time (s)")
    if ch_idx % 4 == 0:
        ax.set_ylabel("Amplitude (z)")
axes[0, 0].legend(fontsize=8, loc="upper left")
fig.suptitle("ERP Comparison: Focused vs Zoned-Out", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("figures/fig3_erp_focused_vs_zoned.png", dpi=200, bbox_inches="tight")
plt.show()